# B2 · Xcorr / franjas

**Spec:** [`docs/spec_B2_codex_xcorr_stripes.md`](../docs/spec_B2_codex_xcorr_stripes.md)  |  **Bloque:** B · Preparación  |  **Run por defecto:** `ROXs12b_realigned`

Correlación cruzada entre exposiciones y detección/corrección de franjas.

| | |
|---|---|
| **Entrada** | Cubo alineado |
| **Salida (QC/productos)** | `stages/stage02_qc.json` (+ `stage02_xcorr_qc.json`) |
| **Consume aguas abajo** | C1, 04b |


## Qué hace B2 y por qué

B2 hace dos cosas sobre el cubo alineado: **correlación cruzada** entre exposiciones (para un shift espectral relativo) y **detección/corrección de franjas** (*stripes*): un patrón aditivo coherente del slicer/detector IFU que puede dejar una modulación en el campo. La maquinaria vive en `musepipe/stripes.py` (diagnosticada en 02b/02c); B2 añade una **métrica escalar de franjas al QC**, propaga STAT por la corrección, y migra el driver — no reimplementa la corrección.

**En este run:** cubo de **exposición única** (`cube_shape[0]=1`) → no hay shift inter-exposición que corregir, y la métrica de franjas sale **plana** (amplitud mediana ~0.977, `reduction_factor = 1.0`, **0 canales sucios**) → **no se aplica corrección de franjas**. Status **green**.

**Para qué sirve aguas abajo:** E2 usa la métrica de franjas para comprobar que cualquier 'señal' de Hα no coincida con un canal de franja residual; D1 compara métodos en canales limpios vs sucios. Aquí, 0 canales sucios → todo limpio. La ventana del láser AO (5780–6050 Å) se excluye del resumen.


## Cómo ejecutar de forma independiente

```bash
conda activate MUSE               # kernel/env con astropy + musepipe
export RUN=ROXs12b_realigned   # o ROXs12b_B_adp para comparar
cd MUSE-accretion-pipeline                    # raíz del repo
bash scripts/stage02_xcorr.sh --run-id $RUN
```

Ligero.

La celda de abajo hace lo mismo desde el notebook (guardada por `RUN`).


In [ ]:
import os, sys
# Añade notebooks/ (para _nbcommon) y la RAÍZ del repo (para importar musepipe),
# funcione el cwd en notebooks/ o en la raíz del repo.
_here = os.getcwd()
if os.path.basename(_here) != 'notebooks' and os.path.isdir(os.path.join(_here, 'notebooks')):
    _here = os.path.join(_here, 'notebooks')
for _p in (_here, os.path.dirname(_here)):
    if _p not in sys.path:
        sys.path.insert(0, _p)
import _nbcommon as nb
_root = str(nb.project_root())
if _root not in sys.path:
    sys.path.insert(0, _root)   # asegura 'import musepipe'
RUN_ID = nb.resolve_run_id(None)
print('run  =', RUN_ID)
print('root =', _root)
print('dir  =', nb.run_dir(RUN_ID))


## Ejecutar o auditar


In [ ]:
RUN = False   # -> True para RE-EJECUTAR esta etapa (regenera su QC)

if RUN:
    cmd = 'bash scripts/stage02_xcorr.sh --run-id $RUN'.replace('$RUN', RUN_ID)
    print('ejecutando:', cmd)
    import subprocess
    subprocess.run(cmd, shell=True, cwd=str(nb.project_root()), check=True)
else:
    print('Modo auditoría (RUN=False): se carga el QC existente abajo.')


## QC / resultados


In [ ]:
qc = nb.load_qc('stages/stage02_xcorr_qc.json', RUN_ID)
nb.show(qc, keys=['status', 'reduction_factor', 'dirty_channels', 'finite_fraction', 'mean_shift'], title='B2')


## Resultados que llevaron a la conclusión

Métrica de franjas + shifts del `stage02_xcorr_qc.json`.


In [ ]:
q = nb.load_qc('stages/stage02_xcorr_qc.json', RUN_ID)
sm = q['stripe_metric']
print('status:', q['status'])
print(f"n_cubes (cube_shape[0]) = {q['cube_shape'][0]}   apply_mode = {q['apply_mode']}")
print(f"shift medio/std por canal = {q['mean_shift_per_cube_ch'][0]} / {q['std_shift_per_cube_ch'][0]}")
print(f"franjas: amp mediana pre={sm['amp_pre_median']:.3f} post={sm['amp_post_median']:.3f}  "
      f"reduction_factor={sm['reduction_factor']:.2f}")
print(f"canales sucios = {len(sm['dirty_channels'])}   fracción finita = {q['finite_fraction_per_cube'][0]:.3f}")
st = q['stat']
print(f"STAT: present={st['present']} shift_applied={st['shift_applied']} kernel={st['interp_kernel']}")
print()
print('provenance:', q['provenance_note'])


## Plot — amplitud del patrón de franjas vs λ

**Tabla usada:** `tables/stage02_stripe_metric.csv` (amplitud por canal, pre y post; aquí **pre ≡ post** porque no se corrigió nada). Archivo pequeño — no hace falta el cubo. La amplitud se mantiene **plana ~0.977** en todo λ (sin patrón coherente de franjas); la ventana del láser AO queda excluida (gris).


In [ ]:
MAKE_PLOT = True   # tabla pequeña; requiere kernel MUSE (pandas/matplotlib)
if MAKE_PLOT:
    try:
        import pandas as pd
        import matplotlib.pyplot as plt
        q = nb.load_qc('stages/stage02_xcorr_qc.json', RUN_ID)
        sm = q['stripe_metric']
        d = pd.read_csv(nb.run_dir(RUN_ID) / 'tables' / 'stage02_stripe_metric.csv')
        w = d['wavelength_A'].values; a = d['amplitude_pre'].values

        fig, ax = plt.subplots(figsize=(11, 4))
        ax.plot(w, a, lw=0.4, color='0.4', label='amplitud de franjas (pre ≡ post)')
        ax.axhline(sm['amp_pre_median'], color='tab:blue', ls='--', lw=1,
                   label=f"mediana={sm['amp_pre_median']:.3f}")
        for i, (x0, x1) in enumerate(sm.get('mask_excluded_windows_A', [])):
            ax.axvspan(x0, x1, color='0.7', alpha=0.4, label='excluido (láser AO)' if i == 0 else None)
        ax.set_xlabel('λ [Å]'); ax.set_ylabel('amplitud del patrón de franjas'); ax.set_ylim(0, 1.6)
        ax.set_title(f"B2 · sin franjas: reduction_factor={sm['reduction_factor']:.2f}, "
                     f"dirty_channels={len(sm['dirty_channels'])} (status {q['status']})")
        ax.legend(fontsize=8); fig.tight_layout()
        outdir = nb.run_dir(RUN_ID) / 'plots' / 'b2_stripes'; outdir.mkdir(parents=True, exist_ok=True)
        fig.savefig(outdir / 'stripe_amplitude.png', dpi=110)
        print('figura ->', outdir / 'stripe_amplitude.png'); plt.show()
    except Exception as e:
        print('No se pudo generar el plot:', type(e).__name__, e)


## Decisiones y notas
- **Exposición única → sin corrección de franjas:** `reduction_factor = 1.0`, 0 canales sucios, shifts = 0; los chequeos de franjas/equivalencia se satisfacen trivialmente.
- Maquinaria de franjas centralizada en `musepipe/stripes.py` (diagnóstico en 02b/02c); B2 solo añade la **métrica escalar** (para E2/D1), propaga STAT y migra el driver.
- Ventana del **láser AO (5780–6050 Å) excluida** del resumen de franjas.


## Conclusión (registrada)

**B2: sin franjas que corregir; status `green`.**

- **Fecha:** run realineado (cubo 2026-07-08; QC 2026-07-08).
- **Datos:** cubo de exposición única; amplitud de franjas mediana **0.977**, `reduction_factor = 1.0`, **0 canales sucios**, fracción finita 0.94.
- **Shifts:** medio/std por canal = 0/0 (nada que alinear entre exposiciones).
- **STAT:** presente, `shift_applied = True`, kernel `none` (sin shift fraccional → sin cambio de varianza).
- **Downstream:** E2 y D1 consumen la métrica de franjas (0 sucios → limpio); el cubo alimenta C1 y 04b.
- **Caveat:** al ser exposición única, los chequeos de franjas/equivalencia son trivialmente satisfechos (open_issue documentado).
